# Sentimiento financiero con FinBERT (y fallback multilingüe)

Este notebook puntúa titulares (o textos cortos) usando modelos de *Transformers* orientados a finanzas.

Salida:
- `sentiment_rows_<TAG>.parquet`: una fila por noticia
- `sentiment_daily_<TAG>.parquet`: agregación diaria por fuente y total


In [ ]:
# Configuracion
from pathlib import Path
import pandas as pd
import numpy as np

DATA_DIR = Path("data")
OUT_DIR = DATA_DIR / "sentiment"
OUT_DIR.mkdir(parents=True, exist_ok=True)

NEWS_DIR = DATA_DIR / "news_daily_mediacloud"
NEWS_GLOB = "news_bitcoin_*.parquet"

# Modelos
# "finbert": ProsusAI/finbert
PRIMARY_MODEL = "finbert"

# Fallback para no-inglés:
# None: se filtra a solo noticias en ingles
# "multilingual": usa cardiffnlp/twitter-xlm-roberta-base-sentiment
FALLBACK_NON_EN = None

# Etiqueta para nombres de salida de ficheros
TAG = f"{PRIMARY_MODEL}" if FALLBACK_NON_EN is None else f"{PRIMARY_MODEL}_fb_{FALLBACK_NON_EN}"

# Ajustes de inferencia
BATCH_SIZE = 16 # Titulares procesados por tanda
MAX_LEN = 192 # Longitud maxima de texto

news_files = sorted(NEWS_DIR.glob(NEWS_GLOB))
print("News files:", len(news_files))
if news_files:
    print("Ejemplo:", news_files[0])
print("TAG:", TAG)

In [ ]:
# Carga de datos
def to_day(s: pd.Series) -> pd.Series:
    """
    Convierte una serie de fechas (str/fecha) a datetime (sin tz) normalizado a día.
    """
    return pd.to_datetime(s, errors="coerce").dt.normalize()

if not news_files:
    raise FileNotFoundError(
        f"No se han encontrado parquets en {NEWS_DIR} con patrón {NEWS_GLOB}. "
    )

dfs = []

for fp in news_files:
    d = pd.read_parquet(fp)[["date", "source", "title", "url"]].copy()

    # Derivados necesarios para el resto del notebook
    d["date"] = to_day(d["date"]) # datetime sin tz (día)
    d["lang"] = "unk"  # luego se detecta si hace falta

    dfs.append(d)

df_all = pd.concat(dfs, ignore_index=True)

print("Total noticias procesadas:", len(df_all))

# Total de noticias por fuente
print("Total de noticias por fuente (Top 20):")
display(df_all["source"].value_counts().head(20))
df_all.head()


In [ ]:
# Detección de idioma
from langdetect import detect

def detect_lang_safe(text):
    try:
        return detect(text)
    except Exception:
        return "unk"

mask_unk = df_all["lang"].isna() | (df_all["lang"].astype(str).str.lower().isin(["unk", "unknown",""]))
if mask_unk.any():
    df_all.loc[mask_unk, "lang"] = df_all.loc[mask_unk, "title"].astype(str).map(detect_lang_safe)

print("lang value counts (tras detectar):")
display(df_all["lang"].value_counts(dropna=False).head(20))

In [ ]:
# Cargar modelos Transformers (FinBERT + multilingüe)
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

MODELS = {
    # FinBERT (finanzas)
    "finbert": "ProsusAI/finbert",
    # Fallback multilingüe
    "multilingual": "cardiffnlp/twitter-xlm-roberta-base-sentiment",
}

def load_model(model_key: str):
    model_id = MODELS[model_key]
    tok = AutoTokenizer.from_pretrained(model_id)
    mod = AutoModelForSequenceClassification.from_pretrained(model_id).to("cpu")
    mod.eval()
    id2label = {int(k): str(v).lower() for k, v in mod.config.id2label.items()}
    return tok, mod, model_id, id2label

tok_primary, mod_primary, model_id_primary, id2label_primary = load_model(PRIMARY_MODEL)
print("PRIMARY:", PRIMARY_MODEL, "->", model_id_primary)

if FALLBACK_NON_EN is not None:
    tok_fb, mod_fb, model_id_fb, id2label_fb = load_model(FALLBACK_NON_EN)
    print("FALLBACK:", FALLBACK_NON_EN, "->", model_id_fb)
else:
    tok_fb = mod_fb = model_id_fb = id2label_fb = None


In [ ]:
# Inferencia batched
from typing import List, Tuple

def softmax_np(x):
    x = x - np.max(x, axis=1, keepdims=True)
    e = np.exp(x)
    return e / np.sum(e, axis=1, keepdims=True)

def score_batch(texts: List[str], tok, mod, id2label: dict, batch_size: int = 16, max_len: int = 192) -> Tuple[list, list, list]:
    """Devuelve label, prob(label), score = P(pos)-P(neg)"""
    labels, probs, scores = [], [], []
    for i in range(0, len(texts), batch_size):
        chunk = texts[i:i+batch_size]
        enc = tok(
            chunk, padding=True, truncation=True, max_length=max_len,
            return_tensors="pt"
        )
        enc = {k: v.to("cpu") for k, v in enc.items()}
        with torch.no_grad():
            logits = mod(**enc).logits.detach().cpu().numpy()

        p = softmax_np(logits)

        # Se intenta detectar ids por label:
        inv = {v: k for k, v in id2label.items()}
        def find_key(cands):
            for c in cands:
                if c in inv:
                    return inv[c]
            return None
        k_pos = find_key(["positive","pos"])
        k_neg = find_key(["negative","neg"])
        k_neu = find_key(["neutral","neu"])

        # Si no las encuentra, se usa argmax como label y score solo con (pos-neg) si se puede
        idx = np.argmax(p, axis=1)

        for j in range(len(chunk)):
            lbl = id2label.get(int(idx[j]), str(idx[j]))
            pr = float(p[j, int(idx[j])])
            # score: P(pos)-P(neg)
            sc = np.nan
            if k_pos is not None and k_neg is not None:
                sc = float(p[j, k_pos] - p[j, k_neg])
            labels.append(lbl.upper())
            probs.append(pr)
            scores.append(sc)
    return labels, probs, scores

def score_dataframe(df: pd.DataFrame, model_key: str, tok, mod, id2label, batch_size: int, max_len: int):
    texts = df["title"].astype(str).tolist()
    labels, probs, scores = score_batch(texts, tok, mod, id2label, batch_size=batch_size, max_len=max_len)
    out = df.copy()
    out["sentiment_label"] = labels
    out["sentiment_prob"] = probs
    out["sentiment_score"] = scores
    out["model_used"] = MODELS[model_key]
    return out


In [ ]:
#Aplicar scoring: inglés con modelo financiero, resto con fallback o filtro
df_all["lang"] = df_all["lang"].astype(str).str.lower()

mask_en = df_all["lang"].str.startswith("en")
df_en = df_all[mask_en].copy()
df_other = df_all[~mask_en].copy()

print("en:", len(df_en), "| other:", len(df_other))

scored_parts = []

if len(df_en):
    scored_en = score_dataframe(df_en, PRIMARY_MODEL, tok_primary, mod_primary, id2label_primary, BATCH_SIZE, MAX_LEN)
    scored_parts.append(scored_en)

if len(df_other):
    if FALLBACK_NON_EN is None:
        #Se descartan del análisis al no perder mucha cobertura de noticias (182 solo)
        print("FALLBACK_NON_EN=None => filtrando a solo inglés. Se descartan:", len(df_other))
    else:
        scored_other = score_dataframe(df_other, FALLBACK_NON_EN, tok_fb, mod_fb, id2label_fb, BATCH_SIZE, MAX_LEN)
        scored_parts.append(scored_other)

df_scored = pd.concat(scored_parts, ignore_index=True)

print("Scored:", df_scored["sentiment_score"].notna().sum(), "/", len(df_scored))
display(df_scored["model_used"].value_counts(dropna=False))
display(df_scored["sentiment_label"].value_counts(dropna=False).head(10))
df_scored.head(3)


In [ ]:
# Agregación diaria (por fuente y total)
daily = (
    df_scored.groupby(["date", "source"], as_index=False)
      .apply(lambda g: pd.Series({
          "n_rows": len(g),
          "n_scored": g["sentiment_score"].notna().sum(),
          "sentiment_mean": g["sentiment_score"].mean(skipna=True),
          "sentiment_median": g["sentiment_score"].median(skipna=True),
      }))
      .reset_index(drop=True)
)

# Total diario (ALL)
daily_all = (
    df_scored.groupby(["date"], as_index=False)
      .apply(lambda g: pd.Series({
          "source": "ALL",
          "n_rows": len(g),
          "n_scored": g["sentiment_score"].notna().sum(),
          "sentiment_mean": g["sentiment_score"].mean(skipna=True),
          "sentiment_median": g["sentiment_score"].median(skipna=True),
      }))
      .reset_index(drop=True)
)

daily_out = (
    pd.concat([daily, daily_all], ignore_index=True)
      .sort_values(["date", "source"])
      .reset_index(drop=True)
)

daily_out.head(5)

In [ ]:
# Guardar salidas
rows_path = OUT_DIR / f"sentiment_rows_{TAG}.parquet"
daily_path = OUT_DIR / f"sentiment_daily_{TAG}.parquet"

# Guardamos la columna 'date' como fecha (YYYY-MM-DD)
df_scored_out = df_scored.copy()
daily_out_out = daily_out.copy()

for _df in (df_scored_out, daily_out_out):
    if "date" in _df.columns:
        _df["date"] = pd.to_datetime(_df["date"], errors="coerce").dt.strftime("%Y-%m-%d")

df_scored_out.to_parquet(rows_path, index=False)
daily_out_out.to_parquet(daily_path, index=False)

print("OK guardado:")
print("-", rows_path)
print("-", daily_path)